# TRACK B — cross-system robustness

**B1 · E-VAE-SCREEN** — VAE forward passes only, no training, ~40 min.
**B2 · E4** — 6 adapters on the selected system, ~4.5 h.

### Pre-registered selection rule — fixed before any generation result is examined
1. Measure the **device-specific paired contrast** R_VAE_post / R_VAE_pre on the H split, the same
   statistic as S1b — not the scalar retention.
2. **Exclude** any model whose post-reconstruction same-model AUC < 0.75.
3. Among survivors, take the **highest device-specific retention η**.
4. **Report every model screened, including the rejects.**

Do not default to SD-1.5: a tighter autoencoder makes the null easier and proves nothing.

### Honest caveat on B2
The training loop here covers the **UNet family** (SD-1.5, SDXL) and reuses the validated SD-3.5
path for the incumbent. **If FLUX wins the screen, B2 needs a new training loop** — its
architecture and LoRA targets differ, and I would not run that blind. Screen first, then decide.

**This is a cross-system robustness test, never an architecture ablation** — VAE, backbone, latent
dimensionality, text encoder, implementation and scale all co-vary.

In [ ]:
# %% B0 — setup
import os, sys, json, glob, math, time, gc, hashlib, subprocess, random
from concurrent.futures import ThreadPoolExecutor
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"]=(userdata.get("HF_TOKEN") or "").strip()
except Exception: pass
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
DRIVE="/content/drive/MyDrive"
BASE=os.path.join(DRIVE,"inv_channel/E_INV_P0_v3")
EXT =os.path.join(DRIVE,"inv_channel/E_TRACKB")
for d in ["","fingerprints","adapters","gens","csv","meta","train_png"]:
    os.makedirs(os.path.join(EXT,d),exist_ok=True)
CACHE="/content/tb_cache"; os.makedirs(CACHE,exist_ok=True)

subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","diffusers>=0.31",
                "transformers","peft","accelerate","safetensors","sentencepiece","protobuf"],check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy import stats as sps
try:
    import peft.import_utils as _piu; _piu.is_torchao_available=lambda: False
    import peft.tuners.lora.torchao as _plt; _plt.is_torchao_available=lambda: False
except Exception: pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])
DEV="cuda" if torch.cuda.is_available() else "cpu"
DT=torch.bfloat16 if DEV=="cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
np.seterr(divide="raise",invalid="raise")
MEAS=1024; _MD=DEV
def HALT(m): raise SystemExit(f"\n[HALT] {m}")
print("device:",DEV,"dtype:",DT)

CANDIDATES = {
 "SD-1.5":        ("stable-diffusion-v1-5/stable-diffusion-v1-5", "vae", "unet"),
 "SDXL":          ("stabilityai/stable-diffusion-xl-base-1.0",    "vae", "unet"),
 "SD-3.5-medium": ("stabilityai/stable-diffusion-3.5-medium",     "vae", "sd3"),
 "FLUX.1-dev":    ("black-forest-labs/FLUX.1-dev",                "vae", "flux"),
 "PixArt-Sigma":  ("PixArt-alpha/pixart_sigma_sdxlvae_T5_diffusers","vae","pixart"),
}
AUC_GATE=0.75

In [ ]:
# %% forensic core (identical settings to the study)
def _cp(fp): return os.path.join(CACHE,hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16]+".npy")
def load_lum(fp,cache=True):
    if cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    l=(0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)
    if cache: np.save(_cp(fp),l)
    return l
def load_rgb(fp):
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        return np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
def _c2(x,k):
    xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
    if _MD=="cuda": return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()
def wres(img):
    s0=np.float32(4.0)
    co=pywt.wavedec2(img,"db8",level=4,mode="periodization"); out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        bnd=[]
        for ch in (cH,cV,cD):
            ch=ch.astype(np.float32); vm=None
            for w in (3,5,7,9):
                v=np.maximum(_c2(ch*ch,np.ones((w,w),np.float32)/(w*w))-s0,0.0)
                vm=v if vm is None else np.minimum(vm,v)
            bnd.append(ch*(s0/(vm+s0)))
        out.append(tuple(bnd))
    n=pywt.waverec2(out,"db8",mode="periodization")[:img.shape[0],:img.shape[1]]
    r=n-n.mean(); sd=float(r.std())
    if not np.isfinite(sd) or sd==0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)
def ncc(a,b):
    a=a-a.mean(); b=b-b.mean()
    na=float(np.linalg.norm(a)); nb=float(np.linalg.norm(b))
    return 0.0 if na==0 or nb==0 else float((a*b).sum()/(na*nb))
def nccb(a,B):
    a=a-a.mean(); na=a.norm()
    Bc=B-B.mean(dim=(1,2),keepdim=True); nb=Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)
import csv as _csv
MAN={}
for r in _csv.DictReader(open(os.path.join(BASE,"manifest","manifest.csv"))):
    MAN.setdefault((r["role"],r["split"]),[]).append(r["path"])
KB={r:np.load(os.path.join(BASE,"fingerprints",f"K_{r}_E2.npy")) for r in ("A","B")}
print("manifest + fingerprints loaded; H split:",len(MAN[("A","H")]),len(MAN[("B","H")]))

In [ ]:
# %% B1 — VAE screen. Reports every candidate, including failures.
def screen_one(name, repo, sub):
    from diffusers import AutoencoderKL
    vae=AutoencoderKL.from_pretrained(repo,subfolder=sub,torch_dtype=torch.float32,
                                      token=True).to(DEV).eval()
    sf=float(vae.config.scaling_factor); rows=[]
    for role in ("A","B"):
        for i,fp in enumerate(MAN[(role,"H")]):
            rgb=load_rgb(fp)
            Y=(0.299*rgb[...,0]+0.587*rgb[...,1]+0.114*rgb[...,2]).astype(np.float32)
            with torch.no_grad():
                x=torch.from_numpy(rgb/127.5-1.0).permute(2,0,1)[None].to(DEV)
                rec=vae.decode(vae.encode(x).latent_dist.mode()).sample[0].permute(1,2,0).cpu().numpy()
            R=np.clip((rec+1.0)*127.5,0,255).astype(np.float32)
            Yr=(0.299*R[...,0]+0.587*R[...,1]+0.114*R[...,2]).astype(np.float32)
            for stage,img in (("pre",Y),("post",Yr)):
                W=wres(img)
                rows.append((role,stage,i,ncc(W,img*KB["A"]),ncc(W,img*KB["B"])))
    del vae; gc.collect(); torch.cuda.empty_cache()
    d=pd.DataFrame(rows,columns=["role","stage","idx","rho_A","rho_B"])
    def contrast(role,stage):
        g=d[(d.role==role)&(d.stage==stage)].sort_values("idx")
        own,oth=("rho_A","rho_B") if role=="A" else ("rho_B","rho_A")
        return g[own].to_numpy()-g[oth].to_numpy()
    out={}
    for stage in ("pre","post"):
        cc=np.r_[contrast("A",stage),contrast("B",stage)]
        same=np.r_[d[(d.role=="A")&(d.stage==stage)].rho_A,d[(d.role=="B")&(d.stage==stage)].rho_B]
        cross=np.r_[d[(d.role=="A")&(d.stage==stage)].rho_B,d[(d.role=="B")&(d.stage==stage)].rho_A]
        lab=np.r_[np.ones(len(same)),np.zeros(len(cross))]; sc=np.r_[same,cross]
        o=np.argsort(sc); rk=np.empty(len(sc)); rk[o]=np.arange(len(sc))
        out[stage]={"contrast":float(cc.mean()),
                    "auc":float((rk[lab==1].mean()-(lab.sum()-1)/2)/(len(sc)-lab.sum()))}
    out["eta_device"]=out["post"]["contrast"]/out["pre"]["contrast"]
    return out, d

SCREEN={}
for name,(repo,sub,fam) in CANDIDATES.items():
    print(f"\n[B1] screening {name} ({repo})")
    try:
        t0=time.time(); r,d=screen_one(name,repo,sub); r["family"]=fam; r["repo"]=repo
        SCREEN[name]=r
        d.to_csv(os.path.join(EXT,"csv",f"b1_{name.replace('.','_').replace('-','_')}.csv"),index=False)
        print(f"   pre  contrast {r['pre']['contrast']:.5e}  AUC {r['pre']['auc']:.4f}")
        print(f"   post contrast {r['post']['contrast']:.5e}  AUC {r['post']['auc']:.4f}")
        print(f"   eta_device = {r['eta_device']:.4f}   ({time.time()-t0:.0f}s)")
    except Exception as e:
        SCREEN[name]={"error":f"{type(e).__name__}: {e}","family":fam,"repo":repo}
        print(f"   FAILED: {type(e).__name__}: {e}")

print("\n"+"="*74)
print(f"{'model':16s} {'family':8s} {'post AUC':>9s} {'eta':>8s} {'gate>=%.2f'%AUC_GATE:>10s}")
elig=[]
for n,r in SCREEN.items():
    if "error" in r: print(f"{n:16s} {r['family']:8s} {'—':>9s} {'—':>8s} {'FAILED':>10s}"); continue
    ok=r["post"]["auc"]>=AUC_GATE
    if ok: elig.append((r["eta_device"],n))
    print(f"{n:16s} {r['family']:8s} {r['post']['auc']:9.4f} {r['eta_device']:8.4f} "
          f"{'PASS' if ok else 'excluded':>10s}")
elig.sort(reverse=True)
incumbent="SD-3.5-medium"
alt=[e for e in elig if e[1]!=incumbent]
WINNER=alt[0][1] if alt else None
print("="*74)
print(f"SELECTED SECOND SYSTEM: {WINNER}" if WINNER else "NO eligible alternative system")
if WINNER and SCREEN[WINNER]["family"]=="flux":
    print("*** FLUX won. B2's training loop does NOT cover it — do not run B2 blind. ***")
json.dump({"screen":SCREEN,"winner":WINNER,"auc_gate":AUC_GATE},
          open(os.path.join(EXT,"B1_screen.json"),"w"),indent=2,default=str)

In [ ]:
# %% B2 — E4 on the selected system. UNet family (SD-1.5 / SDXL) only.
RUN_B2 = False        # set True after reading B1's verdict
N_SEEDS = 3
STEPS, LR, BATCH, GRAD_ACC = 2000, 1e-4, 2, 2
CAPTION = "a photograph, sks style"
GEN_SEED_BASE, G_PER = 770000, 500
UNET_TARGETS = ["to_q","to_k","to_v","to_out.0"]

if RUN_B2:
    sel=json.load(open(os.path.join(EXT,"B1_screen.json")))
    WIN=sel["winner"]; fam=sel["screen"][WIN]["family"]; repo=sel["screen"][WIN]["repo"]
    if fam not in ("unet",): HALT(f"{WIN} is family '{fam}' — this cell covers UNet only. "
                                  f"Write a dedicated loop before running.")
    from diffusers import StableDiffusionPipeline, StableDiffusionXLPipeline, DiffusionPipeline
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    RES = 512 if "1-5" in repo or "1.5" in WIN else 1024
    print(f"[B2] {WIN} ({repo}) family={fam} res={RES}")

    # materialise training crops as PNG at the model's native resolution
    for role in ("A","B"):
        d=os.path.join(EXT,"train_png",f"{role}_raw_{RES}"); os.makedirs(d,exist_ok=True)
        if len(glob.glob(os.path.join(d,"*.png")))>=len(MAN[(role,"T")]): continue
        for i,fp in enumerate(MAN[(role,"T")]):
            a=load_rgb(fp)
            if RES!=MEAS:
                a=np.asarray(Image.fromarray(a.astype(np.uint8)).resize((RES,RES),Image.BICUBIC),np.float32)
            Image.fromarray(a.astype(np.uint8)).save(os.path.join(d,f"{i:04d}.png"),compress_level=1)
        print(f"   materialised {role}: {len(MAN[(role,'T')])} @ {RES}")

    def train(role,seed):
        tag=f"{role}_raw_s{seed}_e4"; out=os.path.join(EXT,"adapters",tag)
        if os.path.exists(os.path.join(out,"pytorch_lora_weights.safetensors")):
            print(f"[B2] skip {tag}"); return tag
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        pipe=DiffusionPipeline.from_pretrained(repo,torch_dtype=DT,token=True).to(DEV)
        sch,vae,unet=pipe.scheduler,pipe.vae,pipe.unet
        with torch.no_grad():
            if hasattr(pipe,"encode_prompt"):
                enc=pipe.encode_prompt(CAPTION,device=DEV,num_images_per_prompt=1,
                                       do_classifier_free_guidance=False)
            else: enc=None
        vae.requires_grad_(False); unet.requires_grad_(False)
        unet.add_adapter(LoraConfig(r=16,lora_alpha=16,init_lora_weights="gaussian",
                                    target_modules=UNET_TARGETS))
        unet.train()
        ps=[p for p in unet.parameters() if p.requires_grad]
        print(f"[B2] {tag}: {sum(p.numel() for p in ps):,} trainable")
        opt=torch.optim.AdamW(ps,lr=LR,weight_decay=1e-4)
        lrs=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(STEPS/GRAD_ACC))
        opt.zero_grad(set_to_none=True)
        files=sorted(glob.glob(os.path.join(EXT,"train_png",f"{role}_raw_{RES}","*.png")))
        cache=torch.stack([torch.from_numpy(
            np.asarray(Image.open(f).convert("RGB"),np.float32)/127.5-1.0).permute(2,0,1) for f in files])
        sf=float(vae.config.scaling_factor)
        g=torch.Generator().manual_seed(seed); losses=[]; t0=time.time()
        emb = enc[0] if isinstance(enc,tuple) else enc
        for st in range(STEPS):
            idx=torch.randint(0,cache.shape[0],(BATCH,),generator=g)
            px=cache[idx].to(DEV,dtype=DT)
            with torch.no_grad(): lat=vae.encode(px).latent_dist.sample()*sf
            noise=torch.randn_like(lat)
            t=torch.randint(0,sch.config.num_train_timesteps,(lat.shape[0],),device=DEV).long()
            noisy=sch.add_noise(lat,noise,t)
            kw={"encoder_hidden_states":emb.repeat(lat.shape[0],1,1).to(DT)}
            pred=unet(noisy,t,**kw).sample
            tgt=noise if sch.config.prediction_type=="epsilon" else sch.get_velocity(lat,noise,t)
            loss=tF.mse_loss(pred.float(),tgt.float())/GRAD_ACC
            loss.backward()
            if (st+1)%GRAD_ACC==0:
                torch.nn.utils.clip_grad_norm_(ps,1.0); opt.step(); lrs.step()
                opt.zero_grad(set_to_none=True)
            losses.append(float(loss.detach())*GRAD_ACC)
            if not np.isfinite(losses[-1]): HALT(f"{tag}: non-finite loss at {st+1}")
            if (st+1)%400==0:
                el=time.time()-t0
                print(f"    {st+1}/{STEPS} loss {np.mean(losses[-400:]):.5f} "
                      f"eta {(el/(st+1))*(STEPS-st-1)/60:.1f}min")
        sd=get_peft_model_state_dict(unet)
        bn=float(sum(float(v.float().norm()) for k,v in sd.items() if "lora_B" in k))
        if not (bn>1e-6): HALT(f"{tag}: DEAD adapter")
        os.makedirs(out,exist_ok=True)
        type(pipe).save_lora_weights(save_directory=out,unet_lora_layers=sd)
        json.dump({"tag":tag,"role":role,"seed":seed,"model":repo,"res":RES,
                   "loss_tail":float(np.mean(losses[-100:])),"lora_B_norm":bn,
                   "minutes":(time.time()-t0)/60},
                  open(os.path.join(out,"train_meta.json"),"w"),indent=2)
        print(f"[B2] saved {tag}: ||lora_B||={bn:.2f} ({(time.time()-t0)/60:.1f} min)")
        del pipe,unet,vae,cache; gc.collect(); torch.cuda.empty_cache()
        return tag

    tags=[train(r,s) for s in range(N_SEEDS) for r in ("A","B")]
    print("\n[B2] adapters:",tags)
    print("[B2] NEXT: generate 500/tag with the paired seed bank, measure against K_A/K_B,")
    print("           then run the adapter-level IUT. Flag the bound as n=3.")
else:
    print("B2 skipped (RUN_B2=False). Read B1's verdict first — if FLUX won, do not run this.")